# DSA 4060: Recommender Systems
## Week 3 Practical Lab: Building a Simple Content-Based Recommender

**Objective:** Build a content-based recommendation system using structured item features, user preference profiles, weighted feature scoring, and recommendation ranking.

# **Step 2- Import Pandas**

In [1]:
import pandas as pd

print("Pandas imported successfully!")

Pandas imported successfully!


# **Step 3 - Create the Movie Dataset**



In [2]:
movies = pd.DataFrame({
    "movie_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "title": [
        "Galaxy Mission", "Love in Nairobi", "Final Battle", "Family Weekend",
        "Future World", "Broken Hearts", "City Detectives", "Space Warriors",
        "Wedding Weekend", "Hidden Truth"
    ],
    "Action":  [1, 0, 1, 0, 1, 0, 1, 1, 0, 0],
    "Comedy":  [0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    "Drama":   [0, 1, 0, 0, 0, 1, 1, 0, 0, 1],
    "Romance": [0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    "SciFi":   [1, 0, 0, 0, 1, 0, 0, 1, 0, 0]
})

movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


# **Step 4- Dataset Inspection**

In [3]:
# Inspect basic information and summary statistics
print("--- Dataset Info ---")
movies.info()

print("\n--- Summary Statistics ---")
movies.describe()

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  10 non-null     int64 
 1   title     10 non-null     object
 2   Action    10 non-null     int64 
 3   Comedy    10 non-null     int64 
 4   Drama     10 non-null     int64 
 5   Romance   10 non-null     int64 
 6   SciFi     10 non-null     int64 
dtypes: int64(6), object(1)
memory usage: 692.0+ bytes

--- Summary Statistics ---


,movie_id,Action,Comedy,Drama,Romance,SciFi
count,10.00000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,5.50000,0.500000,0.200000,0.400000,0.300000,0.300000
std,3.02765,0.527046,0.421637,0.516398,0.483046,0.483046
min,1.00000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.25000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5.50000,0.500000,0.000000,0.000000,0.000000,0.000000
75%,7.75000,1.000000,0.000000,1.000000,0.750000,0.750000
max,10.00000,1.000000,1.000000,1.000000,1.000000,1.000000


###  Interpretation

The dataset has 10 movies and 7 columns with no missing data. It holds a `movie_id`, `title`, and 5 binary (`0`/`1`) genre features. Based on the summary statistics, **Action** appears most frequently (50% of the dataset), followed by **Drama** (40%), **Romance** (30%), **SciFi** (30%), and **Comedy** (20%).

In [4]:
movies.head()
movies.shape
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  10 non-null     int64 
 1   title     10 non-null     object
 2   Action    10 non-null     int64 
 3   Comedy    10 non-null     int64 
 4   Drama     10 non-null     int64 
 5   Romance   10 non-null     int64 
 6   SciFi     10 non-null     int64 
dtypes: int64(6), object(1)
memory usage: 692.0+ bytes


### Step 5 Answers

1. **How many movies are in the dataset?**  
   There are 10 movies in the dataset.

2. **How many genre features exist?**  
   There are 5 genre features (`Action`, `Comedy`, `Drama`, `Romance`, `SciFi`).

3. **What does one row represent?**  
   One row represents a single movie, specified by its unique ID, title, and binary indicators showing which genres it belongs to.

# **Step 6 – Separate the Movie Features**

In [5]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
movie_features = movies[features]
movie_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,1,0,0,0,1
5,0,0,1,1,0
6,1,0,1,0,0
7,1,0,0,0,1
8,0,1,0,1,0
9,0,0,1,0,0


# **Step 7 – Inspect One Item Profile**

In [6]:
movies.loc[movies["title"] == "Galaxy Mission"]

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1


### Step 7 Interpretation

The 1s and 0s in this movie's profile indicate feature presence: a `1` means the movie belongs to that genre, while a `0` means it does not. For *Galaxy Mission*, it has 1s for **Action** and **SciFi**, meaning it is classified under both genres, and 0s for all other genres.

# **Step 8 – Create User Ratings**

In [7]:
# Create User A ratings
ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating": [5, 2, 4, 5]
})
ratings

,movie_id,rating
0,1,5
1,2,2
2,3,4
3,5,5


# **Step 9 – Join Ratings With Movie Features**

In [8]:
# Merge user ratings with the movie feature metadata
user_movies = ratings.merge(movies, on="movie_id")
user_movies

,movie_id,rating,title,Action,Comedy,Drama,Romance,SciFi
0,1,5,Galaxy Mission,1,0,0,0,1
1,2,2,Love in Nairobi,0,0,1,1,0
2,3,4,Final Battle,1,0,0,0,0
3,5,5,Future World,1,0,0,0,1


# **Step 10 – Examine the User's History**

In [9]:
user_movies[["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]]

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,Love in Nairobi,2,0,0,1,1,0
2,Final Battle,4,1,0,0,0,0
3,Future World,5,1,0,0,0,1


### Step 10 Observation

Looking at the user's history, User A heavily favors high-octane sci-fi and action movies, giving 5-star ratings to *Galaxy Mission* and *Future World*, and a 4-star rating to *Final Battle*. On the other hand, they gave a low rating of 2 stars to *Love in Nairobi*, showing little interest in purely drama or romance titles.

# **Step 11 – Select Rated Features**

In [10]:
rated_features = user_movies[features]
user_ratings = user_movies["rating"]

print("--- Rated Features Matrix ---")
print(rated_features)
print("\n--- User Ratings Vector ---")
print(user_ratings)

--- Rated Features Matrix ---
   Action  Comedy  Drama  Romance  SciFi
0       1       0      0        0      1
1       0       0      1        1      0
2       1       0      0        0      0
3       1       0      0        0      1

--- User Ratings Vector ---
0    5
1    2
2    4
3    5
Name: rating, dtype: int64


# **Step 12 – Weight Features Using Ratings**

In [11]:
# Weight each feature vector by the rating given to that movie
weighted_features = rated_features.mul(user_ratings, axis=0)
weighted_features

,Action,Comedy,Drama,Romance,SciFi
0,5,0,0,0,5
1,0,0,2,2,0
2,4,0,0,0,0
3,5,0,0,0,5


### Step 12 Interpretation

Weighting the feature matrix scales the binary indicators by the exact rating assigned to each movie. This ensures that movies the user enjoyed strongly (like 5-star ratings) exert a heavier influence on their overall taste profile than movies they disliked (like 2-star ratings).

# **Step 13 – Build the User Profile**
Sum the weighted features across all watched movies:

In [12]:
# Aggregate feature values to construct the raw user profile vector
user_profile = weighted_features.sum()
user_profile

,0
Action,14
Comedy,0
Drama,2
Romance,2
SciFi,10


# **Step 14 – Interpret the User Profile**
Sort the user profile vector to view the dominant preferences:

In [13]:
user_profile.sort_values(ascending=False)

,0
Action,14
SciFi,10
Drama,2
Romance,2
Comedy,0


### Step 14 Interpretation

The two strongest genres for User A are **Action** (score: 14) and **SciFi** (score: 10). This happens because User A gave high ratings to *Galaxy Mission* (5), *Final Battle* (4), and *Future World* (5), all of which contain Action features (5 + 4 + 5 = 14), while two of them also contain SciFi features (5 + 5 = 10). In contrast, genres like Drama and Romance received a score of only 2, while Comedy sits at 0.

# **Step 15 – Normalize the User Profile**

In [14]:
# Normalize profile vector to represent relative genre proportions
user_profile_normalized = user_profile / user_profile.sum()
user_profile_normalized

,0
Action,0.500000
Comedy,0.000000
Drama,0.071429
Romance,0.071429
SciFi,0.357143


### Step 15 Interpretation

Normalizing the profile converts absolute weighted counts into proportions that sum to 1.0. This makes preference weights comparable across users regardless of how many items they have rated, with Action representing 50.0% of User A's taste profile and SciFi representing ~35.7%.

# **Step 16 – Score Every Movie**

In [15]:
# Calculate dot product score between candidate movies and normalized user profile
movies["recommendation_score"] = (
    movie_features
    .mul(user_profile_normalized)
    .sum(axis=1)
)

movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
1,Love in Nairobi,0.142857
2,Final Battle,0.500000
3,Family Weekend,0.000000
4,Future World,0.857143
5,Broken Hearts,0.142857
6,City Detectives,0.571429
7,Space Warriors,0.857143
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429


# **Step 17 – Rank the Movies**

In [16]:
# Sort candidates in descending order of preference score
ranked_movies = movies.sort_values(
    "recommendation_score",
    ascending=False
)

ranked_movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
4,Future World,0.857143
7,Space Warriors,0.857143
6,City Detectives,0.571429
2,Final Battle,0.500000
1,Love in Nairobi,0.142857
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


# **Step 18 – Remove Movies Already Watched**

In [17]:
# Filter out watched items using boolean indexing
watched_movies = ratings["movie_id"]
recommendations = ranked_movies[
    ~ranked_movies["movie_id"].isin(watched_movies)
]

recommendations[["title", "recommendation_score"]]

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


# **Step 19 – Generate Top 3 Recommendations**

In [18]:
# Extract top 3 candidates
top_3 = recommendations[["title", "recommendation_score"]].head(3)
top_3

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857


# **Step 20 – Explain the Recommendations**


1. **Space Warriors (Score: 0.857):** Recommended first because it combines both **Action** (0.500 weight) and **SciFi** (0.357 weight), perfectly matching User A's two strongest preference features.

2. **City Detectives (Score: 0.571):** Recommended second because it contains **Action** (0.500 weight) alongside Drama (0.071 weight), heavily capitalizing on the user's top preference for action films.

3. **Broken Hearts (Score: 0.143):** Recommended third as a fallback option because it contains Drama and Romance features, capturing the minor residual weights (0.071 + 0.071) present in User A's history.

# **Step 21 – Build a Reusable Function**

In [19]:
def recommend_movies(movies, ratings, features, n=3):
    # Join ratings with movie features
    user_movies = ratings.merge(movies, on="movie_id")

    # Weight features by ratings
    rated_features = user_movies[features]
    weighted_features = rated_features.mul(user_movies["rating"], axis=0)

    # Construct normalized profile vector
    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()

    # Compute candidate scores
    scores = movies[features].mul(user_profile).sum(axis=1)

    # Filter unwatched movies and rank top-N
    results = movies.copy()
    results["recommendation_score"] = scores
    watched = ratings["movie_id"]
    results = results[~results["movie_id"].isin(watched)]

    return results.sort_values("recommendation_score", ascending=False).head(n)

# Test function on User A
recommend_movies(movies, ratings, features)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
7,8,Space Warriors,1,0,0,0,1,0.857143
6,7,City Detectives,1,0,1,0,0,0.571429
5,6,Broken Hearts,0,0,1,1,0,0.142857


# **Step 22 – Test Another User**

In [20]:
# User B ratings: strongly prefers Drama, Romance, and Comedy
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating": [5, 4, 5, 4]
})

recommend_movies(movies, user_b_ratings, features)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
9,10,Hidden Truth,0,0,1,0,0,0.3125
6,7,City Detectives,1,0,1,0,0,0.3125
2,3,Final Battle,1,0,0,0,0,0.0000


### Step 22 Comparison between Users A and B.

User A and User B received entirely distinct recommendations because their underlying rating histories generate fundamentally different taste vectors.

 User A heavily favored Action and SciFi movies, leading to top recommendations like *Space Warriors*.

  In contrast, User B assigned high ratings to Romance, Drama, and Comedy titles (*Love in Nairobi*, *Broken Hearts*, *Family Weekend*), causing the algorithm to prioritize titles with Drama components like *City Detectives* and *Hidden Truth*.

# **Step 23 – Thinking Exercise**



1. **What will the model learn about User C?**  
   The model will learn a hyper-specialized profile based strictly on *Galaxy Mission*, inferring that User C only enjoys Action and Sci-Fi movies.

2. **Which genres will dominate the profile?**  
   **Action** (50%) and **Sci-Fi** (50%) will completely dominate the profile, with zero weight assigned to Comedy, Drama, or Romance.

3. **What could go wrong with the recommendations?**  
   The system falls into a "filter bubble" or over-specialization trap, recommending only Action/Sci-Fi movies and failing to expose the user to other genres they might like.

4. **Is one rating enough to understand a user's preferences?**  
   No. A single rating provides an extremely sparse preference signal that leads to cold-start bias and over-fitting to one specific item's metadata profile.

# **Step 24 – Add Your Own Movie**

In [21]:
# Add a custom, unique movie to the dataset
new_movie = {
    "movie_id": 11,
    "title": "Nairobi Cyberpunk 2088",
    "Action": 1,
    "Comedy": 0,
    "Drama": 0,
    "Romance": 0,
    "SciFi": 1
}

movies = pd.concat([movies, pd.DataFrame([new_movie])], ignore_index=True)
movies.tail()

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
6,7,City Detectives,1,0,1,0,0,0.571429
7,8,Space Warriors,1,0,0,0,1,0.857143
8,9,Wedding Weekend,0,1,0,1,0,0.071429
9,10,Hidden Truth,0,0,1,0,0,0.071429
10,11,Nairobi Cyberpunk 2088,1,0,0,0,1,NaN


# **Step 25 – Create Your Own User**

In [22]:
# Create custom user rating history
my_user = pd.DataFrame({
    "movie_id": [1, 3, 6, 9],
    "rating": [5, 4, 2, 3]
})

# Generate Top 5 recommendations
recommend_movies(movies, my_user, features, n=5)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
4,5,Future World,1,0,0,0,1,0.583333
10,11,Nairobi Cyberpunk 2088,1,0,0,0,1,0.583333
7,8,Space Warriors,1,0,0,0,1,0.583333
6,7,City Detectives,1,0,1,0,0,0.458333
1,2,Love in Nairobi,0,0,1,1,0,0.291667


# **Step 26 – Interpretation Questions**

### Step 26 Interpretation Answers

1. **What are your user's two strongest preferences?**  
   The two strongest preferences are **Action** (weight: 9) and **SciFi** (weight: 5), driven by rating *Galaxy Mission* (5) and *Final Battle* (4) highly.

2. **Which three movies received the highest recommendation scores?**  
   *Future World*, *Space Warriors*, and the newly added *Nairobi Cyberpunk 2088* tied for the top score (~0.583).

3. **Why did the top movie rank first?**  
   They ranked highest because their profiles contain both **Action** and **SciFi**, which directly match the user's highest-weighted genres.

4. **Did any recommendation surprise you?**  
   *Love in Nairobi* appearing at rank 5 with a score of ~0.292 was somewhat surprising given the low rating assigned to *Broken Hearts* (2), but it captured residual weights from Romance/Drama.

5. **What important information is the model ignoring?**  
   The model completely ignores unstructured metadata like director, lead actors, plot nuances, user demographics, release year, and contextual viewing conditions.

# **Step 27 Limitations**

* **Limited Feature Set:** Only five binary genre features are used to describe each movie.
* **Ignored Metadata:** Important movie characteristics such as actors, directors, release year, language, and plot descriptions are completely ignored[cite: 18].
* **Over-Specialization:** Recommendations can become too similar to previously liked items, restricting exposure to new content[cite: 18].
* **User Cold-Start Problem:** A new user with little or no preference history cannot receive accurate recommendations[cite: 18].
* **Metadata Dependency:** The quality of the recommendations depends directly on metadata quality; poor metadata will produce poor recommendations[cite: 18].

# **Step 28 Content-Based Filtering Workflow**

Item Metadata → Item Profiles → User Ratings → User Profile → Candidate Scores → Remove Seen Items → Rank Items → Top-N Recommendations

# **Step 29 Conclusion**

In the lab we managed to construct and assess a content-based recommendation system using structured binary features and user ratings. We determined the candidate similarity scores by weighting the feature vectors with the explicit ratings and normalising the user's taste profile, thereby ranking and recommending movies that the user had not watched according to their tastes.

---

**Final Question Answer**

Question: Should a recommender system provide the same recommendations for two users if those users have watched exactly the same films but have given them different ratings? Use the model from Week 3 to explain your answer.

**Answer:**

The recommender won't give the same recommendations.

In the Week 3 model, the feature vectors are multiplied by each movie’s explicit rating (rated_features.mul(user_ratings, axis=0)) before the values are aggregated and normalized to form a user profile vector.

 Even when both users have watched the same collection of movies, the fact that their rating values differ will cause the underlying genre features to be weighted differently.

 As a result, separate normalized taste profiles ($\mathbf{u}$) are obtained, leading to different dot-product recommendation scores and hence to different candidate rankings.